# 01 — Dados e metadados do vídeo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USUARIO/projeto-plumas/blob/main/01_dados/01_dados.ipynb)

**Objetivo:** documentar o vídeo, preservar o original e extrair quadros com horário em UTC.

**Escopo da professora:** seção 3 (dados e metadados) e seção 5 (manter cópia dos originais).

**Lê:** o vídeo bruto, que você coloca em `<Drive>/01_dados/raw/`.

**Grava:** `metadados.json`, `frames/`, `frames.csv` (e uma cópia pequena de `metadados.json` no repositório).

**GPU:** não


In [ ]:
# === Preparo (igual em todos os notebooks) ===
import os, sys, subprocess

REPO = "USUARIO/projeto-plumas"      # troque pelo seu (veja README)
REPO_DIR = "/content/projeto-plumas"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    try:                                   # repositório privado: secret GITHUB_TOKEN
        from google.colab import userdata
        _tok = userdata.get("GITHUB_TOKEN")
    except Exception:
        _tok = None
    _url = f"https://{_tok}@github.com/{REPO}.git" if _tok else f"https://github.com/{REPO}.git"
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True)
    else:
        _r = subprocess.run(["git", "clone", "-q", _url, REPO_DIR], capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError("Falha no clone. Repositório privado? Configure o secret GITHUB_TOKEN (veja README).")
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:                                      # execução local: sobe até a raiz do repo
    _p = os.getcwd()
    while not os.path.exists(os.path.join(_p, "config.yaml")) and os.path.dirname(_p) != _p:
        _p = os.path.dirname(_p)
    os.chdir(_p)
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from pathlib import Path
from plumas.config import load_config, stage_dir, set_seed, log_experiment
cfg = load_config()
set_seed(cfg["seed"])
STAGE = "01_dados"
D = stage_dir(cfg, STAGE)                  # pasta desta etapa no Drive
print("Etapa:", STAGE, "| dados em:", D)


## 1. Metadados do vídeo
Preencha **todos** os campos obrigatórios. Sem local e horário não há como associar o vídeo ao ERA5,
e sem o azimute da câmera não dá para comparar o deslocamento na imagem com a direção do vento.

- Horário em **UTC** (ISO 8601 com `Z`). Registre também o fuso original, se souber.
- Registre a incerteza da localização (um GPS de celular costuma ter alguns metros; um local aproximado pode ter quilômetros).
- Se o vídeo veio da internet, anote a URL, a data de acesso e a licença/permissão. Confirme com a professora se vídeo público é aceito.

In [ ]:
import json
import pandas as pd
from plumas.config import validate_metadata

meta = {
    "sequence_id": "seq01",            # identificador curto, usado nos nomes dos quadros
    "video_file": "video.mp4",         # nome do arquivo dentro de <Drive>/01_dados/raw/
    "source": "",                      # URL / órgão / autor, ou "gravado por nós"
    "license_or_permission": "",
    "capture_datetime_utc": "",        # início do vídeo, ex.: "2026-11-20T15:30:00Z"
    "original_timezone": "America/Sao_Paulo",
    "latitude": None, "longitude": None,   # graus decimais (câmera ou área observada)
    "location_uncertainty_m": None,
    "camera": "",                      # modelo/tipo; fixa ou móvel
    "resolution": "",                  # ex.: "1920x1080"
    "fps": None,
    "camera_azimuth_deg": None,        # para onde a câmera aponta: 0=N, 90=L (bússola)
    "capture_conditions": "",          # luz, distância à pluma, fundo, clima
    "notes": "",
}
missing = validate_metadata(meta)
print("Campos obrigatórios faltando:", missing or "nenhum")

In [ ]:
meta_path = D / "metadados.json"

def save_meta():
    """Grava no Drive e uma cópia pequena no repositório (versionada no Git)."""
    all_meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
    all_meta[meta["sequence_id"]] = meta
    txt = json.dumps(all_meta, indent=2, ensure_ascii=False)
    meta_path.write_text(txt, encoding="utf-8")
    (Path(cfg["_repo_root"]) / STAGE / "metadados.json").write_text(txt, encoding="utf-8")

assert not missing, f"Preencha antes de continuar: {missing}"
save_meta()
print("Metadados salvos em", meta_path)

## 2. Original intacto
O vídeo fica em `raw/` e **nunca é modificado**. Guardamos o hash SHA-256 para provar, no relatório,
qual arquivo foi usado.

In [ ]:
from plumas.video import sha256_file

raw_dir = D / "raw"
raw_dir.mkdir(exist_ok=True)
video_path = raw_dir / meta["video_file"]
assert video_path.exists(), f"Coloque o vídeo em {video_path}"

meta["sha256"] = sha256_file(video_path)
save_meta()
print("SHA-256:", meta["sha256"])

## 3. Extrair quadros
Um quadro a cada `video.frame_step_s` segundos (veja `config.yaml`). Cada quadro recebe o horário em UTC
(início do vídeo + tempo decorrido). Os quadros extraídos são uma **transformação registrada**: parâmetros no `config.yaml`, log em `runs.jsonl`.

In [ ]:
from plumas.video import extract_frames

frames_dir = D / "frames"
df = extract_frames(video_path, frames_dir, cfg["video"]["frame_step_s"],
                    meta["sequence_id"], meta["capture_datetime_utc"])

csv = D / "frames.csv"
if csv.exists():                               # mantém outras sequências já extraídas
    old = pd.read_csv(csv)
    df = pd.concat([old[old["sequence"] != meta["sequence_id"]], df], ignore_index=True)
df.to_csv(csv, index=False)
print(df.groupby("sequence").size())

In [ ]:
import cv2
import matplotlib.pyplot as plt

sub = df[df["sequence"] == meta["sequence_id"]]
sel = sub.iloc[::max(1, len(sub) // 12)].head(12)
fig, axs = plt.subplots(3, 4, figsize=(16, 9))
for ax in axs.ravel():
    ax.axis("off")
for ax, (_, r) in zip(axs.ravel(), sel.iterrows()):
    ax.imshow(cv2.cvtColor(cv2.imread(str(frames_dir / r["frame"])), cv2.COLOR_BGR2RGB))
    ax.set_title(f'{r["t_s"]:.1f}s', fontsize=9)
plt.tight_layout(); plt.show()

## 4. Conferir o conteúdo do vídeo
- [ ] Há momentos com pluma bem visível **e** momentos com pouca visibilidade?
- [ ] Há estruturas parecidas que **não** são pluma (nuvem, neblina, vapor, poeira)? O escopo pede esses exemplos para medir confusões do modelo.
- [ ] Condições variadas (luz, distância, fundo)? Se não, anote a limitação para o relatório.
- [ ] Mais de uma sequência/local/data? Isso permite divisão por grupo na etapa 02 (veja o que o `config.yaml` pede).

In [ ]:
log_experiment(cfg, STAGE, {
    "sequence": meta["sequence_id"], "video_sha256": meta["sha256"],
    "n_frames": int(len(sub)), "frame_step_s": cfg["video"]["frame_step_s"],
    "decisao": "quadros extraídos; original preservado em raw/",
})